In [ ]:
!pip install -q accelerate lpips

In [ ]:
%%writefile /kaggle/working/finetune_merged.py

# =========================================================================
# FINE-TUNING del modello DDPM MERGIATO con Task Arithmetic
# =========================================================================
# Il modello di partenza è già stato ottenuto fondendo i tre task vector con
# la tecnica Task Arithmetic. Ora viene ripreso quel modello e viene fatta fine-tune
# sull'unione dei dataset con task combinati (es. SR_IR, SR_SU, ecc.), in modo
# che la rete impari a gestire meglio le combinazioni di degradazioni

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import math
import json
import copy
import random
import inspect
from collections import OrderedDict
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset

import matplotlib
matplotlib.use("Agg")  
import matplotlib.pyplot as plt

from accelerate import Accelerator
from accelerate.utils import set_seed as accelerate_set_seed

# LPIPS: usato per lo z-score
try:
    import lpips as _lpips_pkg
    _HAS_LPIPS = True
except Exception:
    _HAS_LPIPS = False


# =========================================================================
# CONFIG
# =========================================================================
# Contiene i path (dei pesi di partenza ottenuti da Task Arithmetic e del dataset 
# dei 7 subset combinati), gli iperparametri di fine-tuning e i
# parametri per il tracking periodico delle metriche immagine durante il training

CONFIG = {
    "merged_init_path": "/kaggle/input/datasets/phantasm04/pesi-dare-ties/ddpm_merged_daretie_final.pth",

    "in_channels": 3,
    "cond_channels": 3,
    "base_channels": 64,
    "n_steps": 200,  

    # --- Dataset: unione dei 7 subset combinati ---
    "dataset_root": "/kaggle/input/datasets/phantasm04/merged/dataset_merged",
    "subsets": ["IR", "IR_SU", "SR", "SR_IR", "SR_IR_SU", "SR_SU", "SU"],

    # Frazione dell'unione da usare e frazione di validazione
    "data_fraction": 0.25,
    "val_fraction": 0.10,
    "split_seed": 42,

    # Manifest JSON con lo split (se esiste viene ricaricato)
    "manifest_path": "/kaggle/working/Risultati_DDPM_MERGED_finetune/split_manifest_merged.json",
    "reuse_manifest": True,

    # --- Iperparametri di fine-tuning  ---
    "finetune_epochs": 25,
    "lr": 2e-5,              
    "weight_decay": 1e-4,
    "batch_size": 8,         
    "grad_clip": 1.0,
    "ema_beta": 0.999,       
    "mixed_precision": "fp16",
    "seed": 42,
    "num_workers": 4,
    "use_compile": False,
    
    "anchor_weight": 0.0,
    "mix_strategy": "balanced",

    "val_max_samples_per_task": 48,
    "track_image_metrics": True,
    "metric_eval_every": 1,            
    "metric_ddim_steps": 10,          
    "metric_max_samples_per_task": 8, 
    "train_eval_max_per_task": 8,      
    "metric_eval_batch_size": 8,
    "lpips_net": "alex",             

    "output_dir": "/kaggle/working/Risultati_DDPM_MERGED_finetune",
    "save_every": 5,
}


# ==================================================================================
# ARCHITETTURA (SelfAttention, UNetBlock, SinusoidalPositionEmbeddings, ImprovedUNet)
# ==================================================================================
# una U-Net condizionale (ImprovedUNet) con blocchi residuali, embedding temporale 
# sinusoidale e self-attention. Prende in input l'immagine rumorosa x_t, il timestep 
# t e l'immagine degradata concatenata come condizione


class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.mha = nn.MultiheadAttention(embed_dim=channels, num_heads=2, batch_first=True)
        self.ln = nn.LayerNorm(channels)
        self.ff_self = nn.Sequential(
            nn.LayerNorm(channels),
            nn.Linear(channels, channels),
            nn.GELU(),
            nn.Linear(channels, channels),
        )

    def forward(self, x):
        size = x.shape[-1]
        x_flat = x.reshape(-1, self.channels, size * size).transpose(1, 2)
        x_ln = self.ln(x_flat)
        attention_value, _ = self.mha(x_ln, x_ln, x_ln)
        attention_value = attention_value + x_flat
        attention_value = self.ff_self(attention_value) + attention_value
        return attention_value.transpose(1, 2).reshape(-1, self.channels, size, size)


class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim=32, num_groups=4):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.act1 = nn.SiLU()
        self.time_mlp = nn.Linear(time_dim, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act2 = nn.SiLU()
        self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, t_emb):
        h = self.act1(self.gn1(self.conv1(x)))
        time_proj = self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = h + time_proj
        h = self.act2(self.gn2(self.conv2(h)))
        return h + self.residual_conv(x)


class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class ImprovedUNet(nn.Module):
    def __init__(self, in_channels=3, cond_channels=3, base_channels=64):
        super().__init__()
        time_dim = base_channels * 4
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(base_channels),
            nn.Linear(base_channels, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, time_dim)
        )
        c = base_channels
        self.inc = UNetBlock(in_channels + cond_channels, c, time_dim)
        self.down1 = nn.Conv2d(c, c, kernel_size=4, stride=2, padding=1)
        self.enc1 = UNetBlock(c, c * 2, time_dim)
        self.down2 = nn.Conv2d(c * 2, c * 2, kernel_size=4, stride=2, padding=1)
        self.enc2 = UNetBlock(c * 2, c * 4, time_dim)
        self.down3 = nn.Conv2d(c * 4, c * 4, kernel_size=4, stride=2, padding=1)
        self.enc3 = UNetBlock(c * 4, c * 8, time_dim)
        self.attn3 = SelfAttention(c * 8)
        self.down4 = nn.Conv2d(c * 8, c * 8, kernel_size=4, stride=2, padding=1)
        self.bottleneck1 = UNetBlock(c * 8, c * 8, time_dim)
        self.attention = SelfAttention(c * 8)
        self.bottleneck2 = UNetBlock(c * 8, c * 8, time_dim)
        self.up1 = nn.ConvTranspose2d(c * 8, c * 8, kernel_size=4, stride=2, padding=1)
        self.dec1 = UNetBlock(c * 16, c * 4, time_dim)
        self.attn_up1 = SelfAttention(c * 4)
        self.up2 = nn.ConvTranspose2d(c * 4, c * 4, kernel_size=4, stride=2, padding=1)
        self.dec2 = UNetBlock(c * 8, c * 2, time_dim)
        self.up3 = nn.ConvTranspose2d(c * 2, c * 2, kernel_size=4, stride=2, padding=1)
        self.dec3 = UNetBlock(c * 4, c, time_dim)
        self.up4 = nn.ConvTranspose2d(c, c, kernel_size=4, stride=2, padding=1)
        self.dec4 = UNetBlock(c * 2, c, time_dim)
        self.out = nn.Conv2d(c, in_channels, kernel_size=3, padding=1)

    def forward(self, x_t, t, condition):
        x_input = torch.cat([x_t, condition], dim=1)
        t_emb = self.time_mlp(t.float())
        s1 = self.inc(x_input, t_emb)
        h = self.down1(s1)
        s2 = self.enc1(h, t_emb)
        h = self.down2(s2)
        s3 = self.enc2(h, t_emb)
        h = self.down3(s3)
        s4 = self.enc3(h, t_emb)
        s4 = self.attn3(s4)
        h = self.down4(s4)
        h = self.bottleneck1(h, t_emb)
        h = self.attention(h)
        h = self.bottleneck2(h, t_emb)
        h = self.up1(h)
        h = torch.cat([h, s4], dim=1)
        h = self.dec1(h, t_emb)
        h = self.attn_up1(h)
        h = self.up2(h)
        h = torch.cat([h, s3], dim=1)
        h = self.dec2(h, t_emb)
        h = self.up3(h)
        h = torch.cat([h, s2], dim=1)
        h = self.dec3(h, t_emb)
        h = self.up4(h)
        h = torch.cat([h, s1], dim=1)
        h = self.dec4(h, t_emb)
        return self.out(h)

# ==================================================================================
# DIFFUSION
# ==================================================================================
# DDPMvPrediction implementa il processo di diffusione con schedule coseno per i beta. 
# Invece di predire il rumore predice la quantità v (ovvero la combinazione di 
# rumore e immagine pulita)


def cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 1e-4, 0.9999)


class DDPMvPrediction(nn.Module):
    def __init__(self, network, n_steps=300):
        super().__init__()
        self.network = network
        self.n_steps = n_steps
        beta = cosine_beta_schedule(n_steps)
        alpha = 1.0 - beta
        alpha_bar = torch.cumprod(alpha, dim=0)
        alpha_bar_prev = torch.cat([torch.tensor([1.0]), alpha_bar[:-1]])
        self.register_buffer('alpha_bar', alpha_bar)
        self.register_buffer('alpha_bar_prev', alpha_bar_prev)
        self.register_buffer('sqrt_alpha_bar', torch.sqrt(alpha_bar))
        self.register_buffer('sqrt_one_minus_alpha_bar', torch.sqrt(1.0 - alpha_bar))
        self.register_buffer('beta', beta)
        self.register_buffer('alpha', alpha)
        self.register_buffer('posterior_variance', beta * (1 - alpha_bar_prev) / (1 - alpha_bar))

    def forward_diffusion(self, x_0, t, noise):
        sqrt_ab = self.sqrt_alpha_bar[t].view(-1, 1, 1, 1)
        sqrt_omab = self.sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1)
        return sqrt_ab * x_0 + sqrt_omab * noise

    def forward(self, x_0, condition):
        batch_size = x_0.shape[0]
        t = torch.randint(0, self.n_steps, (batch_size,), device=x_0.device)
        noise = torch.randn_like(x_0)
        x_t = self.forward_diffusion(x_0, t, noise)
        sqrt_ab = self.sqrt_alpha_bar[t].view(-1, 1, 1, 1)
        sqrt_omab = self.sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1)
        v_target = sqrt_ab * noise - sqrt_omab * x_0
        predicted_v = self.network(x_t, t.float(), condition)
        return F.mse_loss(predicted_v, v_target)

    @torch.no_grad()
    def sample_ddim(self, condition, ddim_steps=25, eta=0.0, clamp_x0=True):
        """Sampler DDIM per v-prediction (deterministico con eta=0)."""
        was_training = self.network.training
        self.network.eval()
        device = condition.device
        B, _, H, W = condition.shape
        x = torch.randn(B, 3, H, W, device=device)
        step_indices = torch.linspace(self.n_steps - 1, 0, ddim_steps).long().to(device)
        for i, t in enumerate(step_indices):
            t_batch = torch.full((B,), int(t), device=device, dtype=torch.long)
            v = self.network(x, t_batch.float(), condition)
            ab_t = self.alpha_bar[t]
            sqrt_ab = ab_t.sqrt()
            sqrt_omab = (1.0 - ab_t).sqrt()
            x0_pred = sqrt_ab * x - sqrt_omab * v
            if clamp_x0:
                x0_pred = x0_pred.clamp(-1.0, 1.0)
            eps_pred = sqrt_omab * x + sqrt_ab * v
            if i == len(step_indices) - 1:
                x = x0_pred
            else:
                t_prev = step_indices[i + 1]
                ab_prev = self.alpha_bar[t_prev]
                sigma = eta * ((1 - ab_prev) / (1 - ab_t)).sqrt() * (1 - ab_t / ab_prev).sqrt()
                dir_coeff = (1.0 - ab_prev - sigma ** 2).clamp(min=0.0).sqrt()
                x = ab_prev.sqrt() * x0_pred + dir_coeff * eps_pred
                if eta > 0:
                    x = x + sigma * torch.randn_like(x)
        if was_training:
            self.network.train()
        return x


class ExponentialMovingAverage:
    def __init__(self, beta=0.999):
        self.beta = beta

    @torch.no_grad()
    def update(self, ema_model, current_model):
        
        ema_params = [p.data for p in ema_model.parameters()]
        cur_params = [p.data for p in current_model.parameters()]
        torch._foreach_mul_(ema_params, self.beta)
        torch._foreach_add_(ema_params, cur_params, alpha=1.0 - self.beta)


def strip_compile_prefix(state_dict):
    if not state_dict:
        return state_dict
    if all(k.startswith("_orig_mod.") for k in state_dict.keys()):
        return {k[len("_orig_mod."):]: v for k, v in state_dict.items()}
    return state_dict


def load_network_state(path):
    """Carica pesi da raw state dict o da checkpoint con 'ema_model'/'model'."""
    ckpt = torch.load(path, map_location="cpu")
    if isinstance(ckpt, dict) and "ema_model" in ckpt:
        state = ckpt["ema_model"]
    elif isinstance(ckpt, dict) and "model" in ckpt:
        state = ckpt["model"]
        
        if any(k.startswith("network.") for k in state.keys()):
            state = {k[len("network."):]: v for k, v in state.items() if k.startswith("network.")}
    else:
        state = ckpt
    return strip_compile_prefix(state)


# =========================================================================
# DATASET
# =========================================================================

class PairedImageDataset(Dataset):
    def __init__(self, condition_paths, target_paths):
        self.condition_paths = list(condition_paths)
        self.target_paths = list(target_paths)
        assert len(self.condition_paths) == len(self.target_paths)
        assert len(self.condition_paths) > 0

    def __len__(self):
        return len(self.condition_paths)

    def __getitem__(self, idx):
        condition_arr = np.load(self.condition_paths[idx])
        target_arr = np.load(self.target_paths[idx])
        condition_tensor = torch.from_numpy(condition_arr).float()
        target_tensor = torch.from_numpy(target_arr).float()
        if condition_tensor.ndim == 3 and condition_tensor.shape[-1] in [1, 3]:
            condition_tensor = condition_tensor.permute(2, 0, 1)
            target_tensor = target_tensor.permute(2, 0, 1)
        condition_tensor = (condition_tensor * 2.0) - 1.0
        target_tensor = (target_tensor * 2.0) - 1.0
        return condition_tensor, target_tensor


# =========================================================================
# SPLIT + MANIFEST
# =========================================================================

def find_pair_dirs(subset_root):
    """Trova le cartelle input/target del subset"""
    candidates = [
        ("input/npy", "target/npy"),
        ("input", "target"),
    ]
    for ci, ct in candidates:
        in_dir = os.path.join(subset_root, ci)
        tgt_dir = os.path.join(subset_root, ct)
        if os.path.isdir(in_dir) and os.path.isdir(tgt_dir):
            return in_dir, tgt_dir
    raise FileNotFoundError(
        f"Nessuna coppia input/target trovata in {subset_root}. "
        f"Contenuto: {os.listdir(subset_root) if os.path.isdir(subset_root) else 'DIRECTORY MANCANTE'}"
    )


def list_pairs(subset_root):
    """Ritorna (in_dir, tgt_dir, [(in_name, tgt_name), ...]) accoppiando i .npy."""
    in_dir, tgt_dir = find_pair_dirs(subset_root)
    in_files = sorted(f for f in os.listdir(in_dir) if f.endswith(".npy"))
    tgt_files = sorted(f for f in os.listdir(tgt_dir) if f.endswith(".npy"))
    common = sorted(set(in_files) & set(tgt_files))
    if len(common) > 0:
        pairs = [(f, f) for f in common]
        dropped = max(len(in_files), len(tgt_files)) - len(common)
        if dropped > 0:
            print(f"[WARN] {subset_root}: {dropped} file senza corrispondenza per basename, scartati.")
    elif len(in_files) == len(tgt_files) and len(in_files) > 0:
        # basename diversi ma stessa cardinalita': accoppia per ordine
        pairs = list(zip(in_files, tgt_files))
        print(f"[WARN] {subset_root}: basename input/target diversi, accoppio per ordinamento.")
    else:
        raise RuntimeError(f"Impossibile accoppiare input/target in {subset_root} "
                           f"(input={len(in_files)}, target={len(tgt_files)}).")
    return in_dir, tgt_dir, pairs


def build_split(config):
    """Crea lo split (frazione dell'unione + train/val per subset) e il manifest."""
    rng = random.Random(config["split_seed"])
    manifest = {
        "created": datetime.now().isoformat(timespec="seconds"),
        "dataset_root": config["dataset_root"],
        "data_fraction": config["data_fraction"],
        "val_fraction": config["val_fraction"],
        "split_seed": config["split_seed"],
        "subsets": {},
    }
    for name in config["subsets"]:
        subset_root = os.path.join(config["dataset_root"], name)
        in_dir, tgt_dir, pairs = list_pairs(subset_root)
        rng.shuffle(pairs)
        n_keep = max(2, int(round(len(pairs) * config["data_fraction"])))
        kept = pairs[:n_keep]
        n_val = max(1, int(round(n_keep * config["val_fraction"])))
        n_val = min(n_val, n_keep - 1)  # almeno 1 campione di train
        val_pairs = kept[:n_val]
        train_pairs = kept[n_val:]
        manifest["subsets"][name] = {
            "input_dir": in_dir,
            "target_dir": tgt_dir,
            "total_available": len(pairs),
            "n_selected": n_keep,
            "train": train_pairs,   # liste [in_name, tgt_name]
            "val": val_pairs,
        }
    return manifest


def load_or_build_manifest(config, accelerator):
    path = config["manifest_path"]
    if config["reuse_manifest"] and os.path.isfile(path):
        with open(path, "r") as f:
            manifest = json.load(f)
        accelerator.print(f"Manifest esistente riusato: {path}")
        return manifest
    manifest = build_split(config)
    if accelerator.is_main_process:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w") as f:
            json.dump(manifest, f, indent=2)
        accelerator.print(f"Nuovo manifest salvato: {path}")
    accelerator.wait_for_everyone()
    return manifest


def datasets_from_manifest(manifest, split):
    """Ritorna {subset_name: PairedImageDataset} per lo split richiesto."""
    out = {}
    for name, info in manifest["subsets"].items():
        pairs = info[split]
        if len(pairs) == 0:
            continue
        cond = [os.path.join(info["input_dir"], p[0]) for p in pairs]
        tgt = [os.path.join(info["target_dir"], p[1]) for p in pairs]
        out[name] = PairedImageDataset(cond, tgt)
    return out


# =========================================================================
# METRICHE (PSNR, MSE, LPIPS)
# =========================================================================

def _to_01(img):
    return (img.clamp(-1.0, 1.0) + 1.0) * 0.5


def psnr_per_sample(pred, target, eps=1e-10):
    p, t = _to_01(pred), _to_01(target)
    mse01 = F.mse_loss(p, t, reduction="none").mean(dim=[1, 2, 3])
    return 10.0 * torch.log10(1.0 / (mse01 + eps))


@torch.no_grad()
def evaluate_image_metrics(ddpm_model, prepared_loaders, accelerator, config, lpips_metric):
    """Per ogni subset calcola MSE (spazio [-1,1], come nel merge), PSNR e,
    per i subset che contengono 'SU', anche LPIPS. Usa sampling DDIM su un
    numero limitato di campioni. Ritorna {subset: {mse, psnr, lpips, n}}."""
    ddpm_model.network.eval()
    cap = config["metric_max_samples_per_task"]
    steps = config["metric_ddim_steps"]
    breakdown = OrderedDict()
    for name, loader in prepared_loaders.items():
        mses, psnrs, lpips_vals, n_seen = [], [], [], 0
        for cond, y in loader:
            pred = ddpm_model.sample_ddim(condition=cond, ddim_steps=steps, eta=0.0)
            mse_s = F.mse_loss(pred, y, reduction="none").mean(dim=[1, 2, 3])
            mses.extend(accelerator.gather_for_metrics(mse_s).cpu().tolist())
            psnrs.extend(accelerator.gather_for_metrics(psnr_per_sample(pred, y)).cpu().tolist())
            if lpips_metric is not None and "SU" in name:
                l_batch = lpips_metric(pred, y).squeeze()
                if l_batch.ndim == 0:
                    l_batch = l_batch.unsqueeze(0)
                lpips_vals.extend(accelerator.gather_for_metrics(l_batch).cpu().tolist())
            n_seen += mse_s.shape[0]
            if cap is not None and n_seen >= cap:
                break
        breakdown[name] = {
            "mse": float(np.mean(mses)) if mses else None,
            "psnr": float(np.mean(psnrs)) if psnrs else None,
            "lpips": float(np.mean(lpips_vals)) if lpips_vals else None,
            "n": len(mses),
        }
    return breakdown


def mean_image_mse(breakdown):
    vals = [v["mse"] for v in breakdown.values() if v.get("mse") is not None]
    return float(np.mean(vals)) if vals else float("nan")


def compute_zscores_over_epochs(breakdowns):
    """z_score standardizzando sulle epoche invece che sui candidati: 
    per ogni subset, LPIPS (se 'SU') e MSE (se 'SR'/'IR') vengono normalizzati 
    con mu/std calcolati su tutta la storia"""
    if not breakdowns:
        return []
    folders = list(breakdowns[0].keys())
    stats = {}
    for f in folders:
        mses = [b[f]["mse"] for b in breakdowns if b[f].get("mse") is not None]
        lps = [b[f]["lpips"] for b in breakdowns if b[f].get("lpips") is not None]
        stats[f] = {
            "mu_mse": np.mean(mses) if mses else 0.0,
            "std_mse": (np.std(mses) + 1e-8) if mses else 1.0,
            "mu_lpips": np.mean(lps) if lps else 0.0,
            "std_lpips": (np.std(lps) + 1e-8) if lps else 1.0,
        }
    zs = []
    for b in breakdowns:
        contribs = []
        for f in folders:
            if "SU" in f and b[f].get("lpips") is not None:
                contribs.append((b[f]["lpips"] - stats[f]["mu_lpips"]) / stats[f]["std_lpips"])
            if ("SR" in f or "IR" in f) and b[f].get("mse") is not None:
                contribs.append((b[f]["mse"] - stats[f]["mu_mse"]) / stats[f]["std_mse"])
        zs.append(float(np.mean(contribs)) if contribs else 0.0)
    return zs


def save_metric_plots(history, plots_dir):
    
    os.makedirs(plots_dir, exist_ok=True)
    epochs = [h["epoch"] for h in history]

    # --- MSE image-space (scala log) ---
    tr_mse = [h.get("train_mean_mse", float("nan")) for h in history]
    va_mse = [h.get("val_mean_mse", float("nan")) for h in history]
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, tr_mse, "-o", label="train", markersize=3)
    plt.plot(epochs, va_mse, "-o", label="val", markersize=3)
    plt.yscale("log")
    plt.xlabel("epoca"); plt.ylabel("MSE media sui subset ([-1,1])  (scala log)")
    plt.title("MSE (image-space) - train vs val")
    plt.grid(alpha=0.3, which="both")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "mse_train_val.png"), dpi=120)
    plt.close()

    # --- z-score con LPIPS (ricalcolato su tutta la storia, traslato e log) ---
    tr_bd = [h["train_breakdown"] for h in history if h.get("train_breakdown")]
    va_bd = [h["val_breakdown"] for h in history if h.get("val_breakdown")]
    tr_z = compute_zscores_over_epochs(tr_bd) if tr_bd else []
    va_z = compute_zscores_over_epochs(va_bd) if va_bd else []
    tr_ep = [h["epoch"] for h in history if h.get("train_breakdown")]
    va_ep = [h["epoch"] for h in history if h.get("val_breakdown")]

    plt.figure(figsize=(8, 5))
    
    # 1. Troviamo il minimo assoluto tra tutti gli z-score calcolati
    all_z = [z for z in (tr_z + va_z) if z is not None and not math.isnan(z)]
    min_z = min(all_z) if all_z else 0.0
    
    # 2. Definiamo un epsilon strettamente positivo per evitare log(0)
    eps = 1e-3 
    
    # 3. Applichiamo lo shift ai dati
    tr_z_shifted = [(z - min_z + eps) for z in tr_z] if tr_z else []
    va_z_shifted = [(z - min_z + eps) for z in va_z] if va_z else []

    if tr_z_shifted:
        plt.plot(tr_ep, tr_z_shifted, "-o", label="train", markersize=3)
    if va_z_shifted:
        plt.plot(va_ep, va_z_shifted, "-o", label="val", markersize=3)

    # Ora possiamo usare una scala logaritmica pura senza errori
    plt.yscale("log")

    plt.xlabel("epoca")
    plt.ylabel("Shifted z-score (LPIPS+MSE)  |  piu' basso = meglio  (scala log)")
    plt.title("Z-score Shiftato con LPIPS - train vs val")
    plt.grid(alpha=0.3, which="both")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "zscore_lpips_train_val.png"), dpi=120)
    plt.close()

    return {"train_z": tr_z, "val_z": va_z}


# =========================================================================
# MAIN
# =========================================================================

def main(config=CONFIG):
    accelerator = Accelerator(mixed_precision=config["mixed_precision"])
    accelerate_set_seed(config["seed"])
    device = accelerator.device
    torch.backends.cudnn.benchmark = True

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

    out_dir = config["output_dir"]
    if accelerator.is_main_process:
        os.makedirs(out_dir, exist_ok=True)
    accelerator.wait_for_everyone()

    # --- 1. split + manifest sull'unione dei 7 subset ---
    manifest = load_or_build_manifest(config, accelerator)
    train_map = datasets_from_manifest(manifest, "train")
    val_map = datasets_from_manifest(manifest, "val")
    for name in train_map:
        accelerator.print(f"[{name:12s}] train={len(train_map[name]):5d}  "
                          f"val={len(val_map[name]) if name in val_map else 0:4d}  "
                          f"(disponibili={manifest['subsets'][name]['total_available']})")

    train_sets = list(train_map.values())
    train_concat = ConcatDataset(train_sets)
    total_train = len(train_concat)

    loader_kwargs = dict(batch_size=config["batch_size"], drop_last=True,
                         num_workers=config["num_workers"], pin_memory=True,
                         persistent_workers=config["num_workers"] > 0,
                         prefetch_factor=4 if config["num_workers"] > 0 else None)

    if config["mix_strategy"] == "balanced":
        
        max_len = max(len(d) for d in train_sets)
        weights = []
        for d in train_sets:
            weights.extend([max_len / len(d)] * len(d))
        sampler = torch.utils.data.WeightedRandomSampler(
            weights=torch.tensor(weights, dtype=torch.double),
            num_samples=total_train,
            replacement=True,
        )
        train_loader = DataLoader(train_concat, sampler=sampler, **loader_kwargs)
    else:
        train_loader = DataLoader(train_concat, shuffle=True, **loader_kwargs)

    val_loaders = {
        name: DataLoader(ds, batch_size=config["batch_size"], shuffle=False, drop_last=False,
                         num_workers=2, pin_memory=True)
        for name, ds in val_map.items()
    }
    
    def _capped_loader(ds, cap):
        n = min(len(ds), cap)
        sub = torch.utils.data.Subset(ds, list(range(n)))
        return DataLoader(sub, batch_size=config["metric_eval_batch_size"],
                          shuffle=False, drop_last=False, num_workers=2, pin_memory=True)

    metric_val_loaders = {name: _capped_loader(ds, config["metric_max_samples_per_task"])
                          for name, ds in val_map.items()}
    metric_train_loaders = {name: _capped_loader(ds, config["train_eval_max_per_task"])
                            for name, ds in train_map.items()}

    # --- 2. modello: init network + EMA dai pesi Task Arithmetic ---
    net = ImprovedUNet(in_channels=config["in_channels"], cond_channels=config["cond_channels"],
                       base_channels=config["base_channels"])
    merged_state = load_network_state(config["merged_init_path"])
    net.load_state_dict(merged_state)
    accelerator.print(f"\nInit dai pesi Task Arithmetic: {config['merged_init_path']}")

    ema_net = copy.deepcopy(net).to(device)
    ema_net.eval()
    for p in ema_net.parameters():
        p.requires_grad_(False)
    ema_updater = ExponentialMovingAverage(beta=config["ema_beta"])

    ddpm = DDPMvPrediction(net, n_steps=config["n_steps"])
    ddpm_ema = DDPMvPrediction(ema_net, n_steps=config["n_steps"]).to(device)
    ddpm_ema.eval()

    if config["use_compile"]:
        try:
            ddpm.network = torch.compile(ddpm.network)
            accelerator.print("torch.compile attivo.")
        except Exception as e:
            accelerator.print(f"torch.compile fallito ({e}), proseguo senza.")

    anchor_weight = config["anchor_weight"]
    anchor_params = None
    if anchor_weight > 0.0:
        anchor_params = [p.detach().clone().to(device) for p in net.parameters()]

    # AdamW fused se supportato (piu' veloce su GPU recenti)
    adamw_kwargs = dict(lr=config["lr"], weight_decay=config["weight_decay"])
    if "fused" in inspect.signature(optim.AdamW).parameters and device.type == "cuda":
        try:
            optimizer = optim.AdamW(ddpm.parameters(), fused=True, **adamw_kwargs)
        except (RuntimeError, ValueError):
            optimizer = optim.AdamW(ddpm.parameters(), **adamw_kwargs)
    else:
        optimizer = optim.AdamW(ddpm.parameters(), **adamw_kwargs)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["finetune_epochs"])

    ddpm, optimizer, train_loader, scheduler = accelerator.prepare(
        ddpm, optimizer, train_loader, scheduler
    )
    prepared_val_loaders = {name: accelerator.prepare(dl) for name, dl in val_loaders.items()}

    track_metrics = config["track_image_metrics"]
    prepared_metric_val = {}
    prepared_metric_train = {}
    lpips_metric = None
    if track_metrics:
        prepared_metric_val = {name: accelerator.prepare(dl) for name, dl in metric_val_loaders.items()}
        prepared_metric_train = {name: accelerator.prepare(dl) for name, dl in metric_train_loaders.items()}
        if _HAS_LPIPS:
            lpips_metric = _lpips_pkg.LPIPS(net=config["lpips_net"]).to(device).eval()
            for p in lpips_metric.parameters():
                p.requires_grad_(False)
            accelerator.print(f"LPIPS ({config['lpips_net']}) pronto per lo z-score.")
        else:
            accelerator.print("[WARN] pacchetto lpips non disponibile: lo z-score "
                              "usera' solo la componente MSE.")

    # --- 3. loop di fine-tuning ---
    best_val = float("inf")
    history = []
    for epoch in range(config["finetune_epochs"]):
        ddpm.train()
        running = 0.0
        n_batches = 0
        for condition, x_0 in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = ddpm(x_0, condition)

            if anchor_params is not None:
                cur = list(accelerator.unwrap_model(ddpm).network.parameters())
                anchor = sum(((p - p0) ** 2).sum() for p, p0 in zip(cur, anchor_params))
                loss = loss + anchor_weight * anchor

            accelerator.backward(loss)
            accelerator.clip_grad_norm_(ddpm.parameters(), max_norm=config["grad_clip"])
            optimizer.step()
            ema_updater.update(ema_net, accelerator.unwrap_model(ddpm).network)

            running += loss.item()
            n_batches += 1

        scheduler.step()
        avg_train = running / max(n_batches, 1)

        # --- validazione PER-SUBSET (con EMA) ---
        ddpm_ema.eval()
        per_task, all_losses = {}, []
        with torch.no_grad():
            for name, loader in prepared_val_loaders.items():
                losses, n_seen = [], 0
                for condition, x_0 in loader:
                    vl = ddpm_ema(x_0, condition)
                    gathered = accelerator.gather_for_metrics(vl.repeat(x_0.shape[0]))
                    losses.extend(gathered.cpu().tolist())
                    n_seen += len(gathered)
                    cap = config["val_max_samples_per_task"]
                    if cap is not None and n_seen >= cap:
                        break
                per_task[name] = sum(losses) / max(len(losses), 1)
                all_losses.extend(losses)
        mean_val = sum(all_losses) / max(len(all_losses), 1)

        # --- metriche MSE e LPIPS su train e val per z-score ---
        train_bd, val_bd = None, None
        train_mean_mse = float("nan")
        val_mean_mse = float("nan")
        do_metrics = track_metrics and ((epoch + 1) % config["metric_eval_every"] == 0
                                        or epoch == config["finetune_epochs"] - 1)
        if do_metrics:
            val_bd = evaluate_image_metrics(ddpm_ema, prepared_metric_val,
                                            accelerator, config, lpips_metric)
            train_bd = evaluate_image_metrics(ddpm_ema, prepared_metric_train,
                                              accelerator, config, lpips_metric)
            val_mean_mse = mean_image_mse(val_bd)
            train_mean_mse = mean_image_mse(train_bd)

        if accelerator.is_main_process:
            bd = " | ".join(f"{k}={v:.4f}" for k, v in per_task.items())
            extra = ""
            if do_metrics:
                extra = f" | MSE(img) tr={train_mean_mse:.4f}/va={val_mean_mse:.4f}"
            accelerator.print(f"Epoca {epoch+1:03d}/{config['finetune_epochs']} | "
                              f"train={avg_train:.4f} | val_mean(EMA)={mean_val:.4f}{extra} | {bd}")
            history.append({
                "epoch": epoch + 1,
                "train": avg_train,
                "val_mean": mean_val,
                "per_task": per_task,
                "train_mean_mse": train_mean_mse,
                "val_mean_mse": val_mean_mse,
                "train_breakdown": train_bd,
                "val_breakdown": val_bd,
            })
            with open(os.path.join(out_dir, "training_history.json"), "w") as f:
                json.dump(history, f, indent=2)

            # plot MSE e z-score-LPIPS aggiornati a ogni epoca
            zinfo = {"train_z": [], "val_z": []}
            if track_metrics:
                zinfo = save_metric_plots(history, os.path.join(out_dir, "plots"))
        accelerator.wait_for_everyone()

        # z-score dell'epoca corrente 
        cur_val_z = zinfo["val_z"][-1] if accelerator.is_main_process and zinfo["val_z"] else None
        cur_train_z = zinfo["train_z"][-1] if accelerator.is_main_process and zinfo["train_z"] else None

        def _save(path, epoch_idx):
            unwrapped = accelerator.unwrap_model(ddpm)
            net_state = strip_compile_prefix(unwrapped.network.state_dict())
            torch.save({
                "epoch": epoch_idx,
                "model": net_state,
                "ema_model": ema_net.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "val_mean": mean_val,
                "per_task_val": per_task,
                "train_mean_mse": train_mean_mse,
                "val_mean_mse": val_mean_mse,
                "train_image_breakdown": train_bd,
                "val_image_breakdown": val_bd,
                "train_z_lpips": cur_train_z,
                "val_z_lpips": cur_val_z,
                "manifest_path": config["manifest_path"],
                "merge_method": "task_arithmetic",
            }, path)

        if accelerator.is_main_process:
            if mean_val < best_val:
                best_val = mean_val
                _save(os.path.join(out_dir, "ddpm_merged_finetuned_best.pth"), epoch)
                accelerator.print(f"  >>> nuovo best val_mean={best_val:.6f}, salvato.")
            if (epoch + 1) % config["save_every"] == 0:
                _save(os.path.join(out_dir, f"ddpm_merged_finetuned_epoch_{epoch+1}.pth"), epoch)
        accelerator.wait_for_everyone()

    if accelerator.is_main_process:
        accelerator.print("\nFine-tuning completato.")
        accelerator.print(f"Best (val_mean={best_val:.6f}): {os.path.join(out_dir, 'ddpm_merged_finetuned_best.pth')}")
        accelerator.print(f"Manifest dello split: {config['manifest_path']}")
        accelerator.print(f"History JSON: {os.path.join(out_dir, 'training_history.json')}")
        if track_metrics:
            accelerator.print(f"Plot MSE:      {os.path.join(out_dir, 'plots', 'mse_train_val.png')}")
            accelerator.print(f"Plot z-score:  {os.path.join(out_dir, 'plots', 'zscore_lpips_train_val.png')}")


if __name__ == "__main__":
    main()

In [ ]:
!accelerate launch --multi_gpu --num_processes=2 --mixed_precision=fp16 /kaggle/working/finetune_merged.py

In [ ]:
%%writefile /kaggle/working/viz_merged.py

# =========================================================================
# VISUALIZZAZIONE del modello DDPM MERGIATO dopo fine-tuning
# =========================================================================

import os
import json

import matplotlib
matplotlib.use("Agg")  
import matplotlib.pyplot as plt
import numpy as np
import torch

# Tutte le classi arrivano dallo script di fine-tuning (stesso /kaggle/working)
from finetune_merged import (
    ImprovedUNet, DDPMvPrediction, PairedImageDataset, load_network_state
)

# =========================================================================
# CONFIGURAZIONE e CARICAMENTO MODELLO
# =========================================================================

MODEL_PATH = "/kaggle/working/Risultati_DDPM_MERGED_finetune/ddpm_merged_finetuned_best.pth"
MANIFEST_PATH = "/kaggle/working/Risultati_DDPM_MERGED_finetune/split_manifest_merged.json"
OUTPUT_DIR = "/kaggle/working/Risultati_DDPM_MERGED_finetune/viz"

N_EXAMPLES_PER_SUBSET = 4   
DDIM_STEPS = 25
N_STEPS = 200               
SEED = 123                 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.manual_seed(SEED)

print(f"Caricamento del modello fuso (Task Arithmetic, fine-tuned) da: {MODEL_PATH}")
unet = ImprovedUNet(in_channels=3, cond_channels=3, base_channels=64)
state_dict = load_network_state(MODEL_PATH) 
unet.load_state_dict(state_dict)
unet.to(DEVICE)
unet.eval()

ddpm = DDPMvPrediction(unet, n_steps=N_STEPS).to(DEVICE)
ddpm.eval()

# =========================================================================
# CARICAMENTO MANIFEST 
# =========================================================================

print(f"Caricamento manifest da: {MANIFEST_PATH}")
with open(MANIFEST_PATH, "r") as f:
    manifest = json.load(f)


def denormalize(t):
    """Riporta i tensori dal range [-1, 1] a [0, 1]."""
    return torch.clamp((t + 1.0) / 2.0, 0.0, 1.0)


def to_img(t):
    """(C,H,W) tensor -> (H,W,C) numpy in [0,1], pronto per imshow."""
    img = denormalize(t).detach().cpu().permute(1, 2, 0).numpy()
    return img.squeeze(-1) if img.shape[-1] == 1 else img


# =========================================================================
# INFERENZA E SALVATAGGIO PER OGNI SUBSET
# =========================================================================

summary = {"model_path": MODEL_PATH, "manifest_path": MANIFEST_PATH,
           "ddim_steps": DDIM_STEPS, "seed": SEED, "subsets": {}}

rng = np.random.default_rng(SEED)

for subset_name, info in manifest["subsets"].items():
    val_pairs = info["val"]
    if len(val_pairs) == 0:
        print(f"[WARN] {subset_name}: nessun campione di validazione, salto.")
        continue

    n = min(N_EXAMPLES_PER_SUBSET, len(val_pairs))
    chosen = sorted(rng.choice(len(val_pairs), size=n, replace=False).tolist())

    cond_paths = [os.path.join(info["input_dir"], val_pairs[i][0]) for i in chosen]
    tgt_paths = [os.path.join(info["target_dir"], val_pairs[i][1]) for i in chosen]
    dataset = PairedImageDataset(cond_paths, tgt_paths)

    conds, targets = zip(*[dataset[i] for i in range(len(dataset))])
    cond_batch = torch.stack(conds).to(DEVICE)     # (n, C, H, W)
    target_batch = torch.stack(targets)

    print(f"[{subset_name}] campionamento DDIM in batch su {n} esempi "
          f"(indici val: {chosen})...")
    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=torch.float16,
                            enabled=(DEVICE == "cuda")):
            preds = ddpm.sample_ddim(condition=cond_batch,
                                     ddim_steps=DDIM_STEPS, eta=0.0)
    preds = preds.float().cpu()

    # --- metriche (in [0,1]) ---
    subset_metrics = []
    for j in range(n):
        p01 = denormalize(preds[j])
        t01 = denormalize(target_batch[j])
        mse = float(torch.mean((p01 - t01) ** 2))
        psnr = float(10.0 * np.log10(1.0 / max(mse, 1e-12)))
        subset_metrics.append({
            "val_index": chosen[j],
            "input_file": val_pairs[chosen[j]][0],
            "mse": mse,
            "psnr_db": psnr,
        })

    # --- figura: input, target, prediction ---
    fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n), squeeze=False)
    for j in range(n):
        axes[j][0].imshow(to_img(conds[j]))
        axes[j][0].set_title(f"Input degradato (val idx {chosen[j]})", fontsize=12)
        axes[j][1].imshow(to_img(target_batch[j]))
        axes[j][1].set_title("Target ground truth", fontsize=12)
        axes[j][2].imshow(to_img(preds[j]))
        axes[j][2].set_title(f"Prediction | PSNR {subset_metrics[j]['psnr_db']:.2f} dB",
                             fontsize=12)
        for k in range(3):
            axes[j][k].axis("off")
    fig.suptitle(f"Subset: {subset_name}  (modello Task Arithmetic fine-tuned)",
                 fontsize=16, y=1.0)
    fig.tight_layout()
    fig_path = os.path.join(OUTPUT_DIR, f"{subset_name}.png")
    fig.savefig(fig_path, dpi=110, bbox_inches="tight")
    plt.close(fig)  # libera memoria: fondamentale con 7 figure in un run
    print(f"[{subset_name}] figura salvata in: {fig_path}")

    summary["subsets"][subset_name] = {
        "figure": fig_path,
        "examples": subset_metrics,
        "mean_mse": float(np.mean([m["mse"] for m in subset_metrics])),
        "mean_psnr_db": float(np.mean([m["psnr_db"] for m in subset_metrics])),
    }

summary_path = os.path.join(OUTPUT_DIR, "metrics_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n================ RIEPILOGO ================")
for name, s in summary["subsets"].items():
    print(f"{name:12s} | PSNR medio: {s['mean_psnr_db']:6.2f} dB | MSE medio: {s['mean_mse']:.5f}")
print(f"\nMetriche salvate in: {summary_path}")
print(f"Figure salvate in:   {OUTPUT_DIR}/<SUBSET>.png")

In [ ]:
!accelerate launch --multi_gpu --num_processes=2 --mixed_precision=fp16 /kaggle/working/viz_merged.py